In [0]:
from pyspark.sql import functions as F

orders_bronze = spark.table(
    "workspace.fashion_bronze.orders_ingested"
)

print("Bronze order records:", orders_bronze.count())
orders_bronze.printSchema()

Bronze order records: 200000
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- gross_order_amount: double (nullable = true)
 |-- discount_order_amount: double (nullable = true)
 |-- net_order_amount: double (nullable = true)
 |-- total_items: integer (nullable = true)
 |-- shipping_city: string (nullable = true)



In [0]:
orders_bronze.select(
    F.count("*").alias("total_orders"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("guest_orders"),
    F.min("order_date").alias("first_order_date"),
    F.max("order_date").alias("last_order_date"),
    F.sum("net_order_amount").alias("total_revenue"),
    F.avg("net_order_amount").alias("average_order_value")
).show()

+------------+-------------+----------------+------------+----------------+---------------+--------------------+-------------------+
|total_orders|unique_orders|unique_customers|guest_orders|first_order_date|last_order_date|       total_revenue|average_order_value|
+------------+-------------+----------------+------------+----------------+---------------+--------------------+-------------------+
|      200000|       200000|           81701|       29882|      2023-01-01|     2026-10-31|2.9353008786999664E9| 14676.504393499832|
+------------+-------------+----------------+------------+----------------+---------------+--------------------+-------------------+



In [0]:
for column in ["order_status", "order_channel", "payment_method", "shipping_city", "coupon_code"]:
    print(f"\n{column.upper()}")
    orders_bronze.groupBy(column) \
        .count() \
        .orderBy(column) \
        .show(truncate=False)


ORDER_STATUS
+------------+------+
|order_status|count |
+------------+------+
|Cancelled   |19930 |
|Delivered   |166872|
|Processing  |810   |
|Returned    |12028 |
|Shipped     |360   |
+------------+------+


ORDER_CHANNEL
+-------------+------+
|order_channel|count |
+-------------+------+
|Mobile App   |80267 |
|Website      |119733|
+-------------+------+


PAYMENT_METHOD
+----------------+-----+
|payment_method  |count|
+----------------+-----+
|Cash on Delivery|33059|
|Credit Card     |33585|
|Debit Card      |33354|
|Net Banking     |33618|
|UPI             |33234|
|Wallet          |33150|
+----------------+-----+


SHIPPING_CITY
+-------------+-----+
|shipping_city|count|
+-------------+-----+
|Bengaluru    |50028|
|Delhi        |50023|
|Mumbai       |49984|
|Pune         |49965|
+-------------+-----+


COUPON_CODE
+-----------+------+
|coupon_code|count |
+-----------+------+
|FESTIVE15  |15938 |
|NONE       |120146|
|NOVA10     |24127 |
|STYLE20    |9900  |
|WELCOME5   |2

In [0]:
print("Total orders:", orders_bronze.count())

print("Duplicate order IDs:",
      orders_bronze.groupBy("order_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

for column in orders_bronze.columns:
    null_count = orders_bronze.filter(
        F.col(column).isNull()
    ).count()
    print(f"{column}: {null_count} null values")

Total orders: 200000
Duplicate order IDs: 0
order_id: 0 null values
order_date: 0 null values
customer_id: 29882 null values
order_channel: 0 null values
payment_method: 0 null values
order_status: 0 null values
coupon_code: 0 null values
gross_order_amount: 0 null values
discount_order_amount: 0 null values
net_order_amount: 0 null values
total_items: 0 null values
shipping_city: 0 null values


In [0]:
print("Invalid order dates:",
      orders_bronze.filter(
          (F.col("order_date") < F.lit("2023-01-01")) |
          (F.col("order_date") > F.lit("2026-10-31"))
      ).count())

print("Non-positive gross amounts:",
      orders_bronze.filter(F.col("gross_order_amount") <= 0).count())

print("Negative discounts:",
      orders_bronze.filter(F.col("discount_order_amount") < 0).count())

print("Negative net amounts:",
      orders_bronze.filter(F.col("net_order_amount") < 0).count())

print("Order total mismatches:",
      orders_bronze.filter(
          F.abs(
              F.col("gross_order_amount") -
              F.col("discount_order_amount") -
              F.col("net_order_amount")
          ) > 0.01
      ).count())

print("Orders with invalid item counts:",
      orders_bronze.filter(F.col("total_items") <= 0).count())

Invalid order dates: 0
Non-positive gross amounts: 0
Negative discounts: 0
Negative net amounts: 0
Order total mismatches: 0
Orders with invalid item counts: 0


In [0]:
orders_silver = (
    orders_bronze
    .withColumn("order_id", F.upper(F.trim(F.col("order_id"))))
    .withColumn("order_channel", F.trim(F.col("order_channel")))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    .withColumn("order_status", F.initcap(F.trim(F.col("order_status"))))
    .withColumn("coupon_code", F.upper(F.trim(F.col("coupon_code"))))
    .withColumn("shipping_city", F.initcap(F.trim(F.col("shipping_city"))))
)

In [0]:
print("Total Silver orders:", orders_silver.count())

print("Duplicate order IDs:",
      orders_silver.groupBy("order_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Invalid order channels:",
      orders_silver.filter(
          ~F.col("order_channel").isin("Website", "Mobile App")
      ).count())

print("Invalid order statuses:",
      orders_silver.filter(
          ~F.col("order_status").isin(
              "Delivered", "Cancelled", "Processing", "Returned", "Shipped"
          )
      ).count())

print("Guest orders preserved:",
      orders_silver.filter(F.col("customer_id").isNull()).count())

Total Silver orders: 200000
Duplicate order IDs: 0
Invalid order channels: 0
Invalid order statuses: 0
Guest orders preserved: 29882


In [0]:
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_silver.orders")

In [0]:
saved_orders = spark.table("workspace.fashion_silver.orders")

print("Saved order records:", saved_orders.count())
saved_orders.printSchema()
saved_orders.show(5, truncate=False)

Saved order records: 200000
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- gross_order_amount: double (nullable = true)
 |-- discount_order_amount: double (nullable = true)
 |-- net_order_amount: double (nullable = true)
 |-- total_items: integer (nullable = true)
 |-- shipping_city: string (nullable = true)

+-----------+----------+-----------+-------------+----------------+------------+-----------+------------------+---------------------+----------------+-----------+-------------+
|order_id   |order_date|customer_id|order_channel|payment_method  |order_status|coupon_code|gross_order_amount|discount_order_amount|net_order_amount|total_items|shipping_city|
+-----------+----------+-----------+-------------+----------------+---